In [ ]:
"""
Custom RMSLE metric for forecasting raw material weights.

This metric expects three DataFrames:
- metadata: maps ID to material ID and forecast time window
- submission: contains predicted weight per ID
- solution: contains actual (true) weight per ID

The metric calculates RMSLE over the predicted and actual weights.

Required columns:
- metadata: ['ID', 'rm_id', 'forecast_start_date', 'forecast_end_date']
- submission: ['ID', 'predicted_weight']
- solution: ['ID', 'weight']
"""

import pandas as pd
import numpy as np

class ParticipantVisibleError(Exception):
    """Raise this for participant-facing errors."""
    pass


def quantile_error(actual: np.ndarray, predicted: np.ndarray, q: float = 0.8) -> float:
    """Quantile loss (pinball loss) for quantile q."""
    if np.any(actual < 0) or np.any(predicted < 0):
        raise ParticipantVisibleError("Values must be non-negative.")

    diff = actual - predicted
    return np.mean(np.maximum(q * diff, (q - 1) * diff))


def score(solution: pd.DataFrame, submission: pd.DataFrame,
          metadata: pd.DataFrame) -> float:
    """
    Compute the 0.8 quantile of underestimation error between predictions and true weights.

    Args:
        solution (pd.DataFrame): True weights per ID. Must have ['ID', 'weight']
        submission (pd.DataFrame): Predicted weights per ID. Must have ['ID', 'predicted_weight']
        metadata (pd.DataFrame): Additional columns ['ID', 'rm_id', 'forecast_start_date', 'forecast_end_date'] (prediction_mapping.csv)

    Returns:
        float: 0.8 quantile of underestimation error
    """

    for col in ['ID', 'predicted_weight']:
        if col not in submission.columns:
            raise ParticipantVisibleError(f"Submission is missing column: {col}")
        
    if not pd.api.types.is_numeric_dtype(submission['predicted_weight']):
        raise ParticipantVisibleError("'predicted_weight' in submission must be numeric.")

    submission_filtered = submission[submission['ID'].isin(solution['ID'])]

    merged = pd.merge(
        solution, 
        submission_filtered, 
        on='ID', 
        how='left', 
        validate='one_to_one'
    )

    if merged['predicted_weight'].isnull().any():
        missing_ids = merged.loc[merged['predicted_weight'].isnull(), 'ID'].tolist()
        raise ParticipantVisibleError(f"Missing predictions for required ID(s): {missing_ids[:5]}{'...' if len(missing_ids) > 5 else ''}")

    actual = merged['weight'].values
    predicted = merged['predicted_weight'].values

    try:
        result = quantile_error(actual, predicted, q=0.8)
    except Exception as e:
        raise ParticipantVisibleError(f"Error during underestimation quantile calculation: {e}")

    if not np.isfinite(result):
        raise ParticipantVisibleError("Final quantile error is not a finite number.")

    return float(result)
